# Update time series

In [ ]:
from __future__ import annotations

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import datetime as dt

import numpy as np
import pandas as pd
import xarray as xr

In [ ]:
from sweat.timeseries import status_handler as sh
from sweat.timeseries.constant import TimeSeriesVar as TSVar
from sweat.timeseries.linear_updater import LinearUpdater

### Initiate data

In [ ]:
# Windows size
window_size = 7
# Dimensions
x_dim = 1
y_dim = 2
time_dim = window_size
# Date time series
today = dt.datetime.now(tz=dt.UTC).date()
# Generate an array of consecutive dates
dates = np.array(pd.date_range(end=today, periods=window_size).to_list())
print(dates)


# Create ET time series
def et_piecewise_function(x):
    return np.where(x < 0, x + 6, -0.5 * x + 6)


# Calculate the corresponding y values using the piecewise function
et = et_piecewise_function(np.linspace(-3, 3, window_size))
et_time_series = et.copy()
print(et_time_series)
et_time_series[-1] = np.nan


# Create ET time series
def radiation_piecewise_function(x):
    return np.where(x < 0, 100 * x + 600, -50 * x + 600)


# Calculate the corresponding y values using the piecewise function
radiation_time_series = radiation_piecewise_function(
    np.linspace(-3, 3, window_size)
)
print(radiation_time_series)

# Flags time series
flags_time_series = np.array(
    [
        0b000000000000000,
        0b000000011000010,
        0b000000011000010,
        0b000000000000000,
        0b000000001000011,
        0b000000010000011,
        0b000000000000001,
    ],
    dtype=np.uint16,
)

In [ ]:
# Create an xarray dataset
ts_ds = xr.Dataset(
    {
        TSVar.ET.value: (
            ["time", "y", "x"],
            np.tile(
                et_time_series[:, np.newaxis, np.newaxis], (1, y_dim, x_dim)
            ),
        ),
        TSVar.RADIATION.value: (
            ["time", "y", "x"],
            np.tile(
                radiation_time_series[:, np.newaxis, np.newaxis],
                (1, y_dim, x_dim),
            ),
        ),
        TSVar.FLAGS.value: (
            ["time", "y", "x"],
            np.tile(
                flags_time_series[:, np.newaxis, np.newaxis], (1, y_dim, x_dim)
            ),
        ),
    },
    coords={
        "time": dates,
        "x": np.arange(x_dim),
        "y": np.arange(y_dim),
    },
)

# Display the dataset
ts_ds

In [ ]:
ts_ds["et"].values

In [ ]:
acquisition_dates = [0, 3, 6]
feed_dates = dates[acquisition_dates]
feed_et = et[acquisition_dates]
valid = np.array([0, 1, 1])

In [ ]:
# Create an xarray dataset
feed_ds = xr.Dataset(
    {
        TSVar.ET.value: (
            ["time", "y", "x"],
            np.tile(feed_et[:, np.newaxis, np.newaxis], (1, y_dim, x_dim)),
        ),
        TSVar.VALID.value: (
            ["time", "y", "x"],
            np.tile(valid[:, np.newaxis, np.newaxis], (1, y_dim, x_dim)),
        ),
    },
    coords={
        "time": feed_dates,
        "x": np.arange(x_dim),
        "y": np.arange(y_dim),
    },
)

# Display the dataset
feed_ds

### Extract data for one pixel

In [ ]:
ts = ts_ds.isel(x=0, y=1)

In [ ]:
ts

In [ ]:
ts[TSVar.ET.value].plot()

In [ ]:
feed = feed_ds.isel(x=0, y=1)

In [ ]:
feed

### LinearUpdater

In [ ]:
linear_updater = LinearUpdater(
    strict_mode=True, radiation_mode=sh.RadiationMode.EXTERNAL
)

In [ ]:
status_arr = ts[TSVar.FLAGS.value].data
status_arr

In [ ]:
linear_updater.find_previous(status_arr, 2)

In [ ]:
linear_updater.find_next(status_arr, 2)

In [ ]:
ts[TSVar.ET.value].data

In [ ]:
linear_updater.interpolate(2, 1, 3, ts)

In [ ]:
linear_updater.extrapolate(3, 2, ts)

In [ ]:
linear_updater.backward_extrapolate(2, 3, ts)

Update new data

In [ ]:
linear_updater.update_new_acquisition(ts, feed)

Update time series

In [ ]:
# Change the values at the specified positions
for pos in [1, 6]:
    ts[TSVar.ET.value][pos] = np.nan
    ts[TSVar.FLAGS.value][pos] = sh.INIT_STATUS

In [ ]:
linear_updater.update_time_series(ts)

Update the dataset

In [ ]:
list(ts_ds.sizes.keys())

In [ ]:
linear_updater.update(ts_ds, feed_ds)